In [76]:
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv


In [44]:
PATH = 'data/earning_call_presentations/clean_tr_sp500_final_2015_2020.parquet'

In [46]:
df = pd.read_parquet(PATH)

In [78]:
load_dotenv()
client = OpenAI()

In [112]:
Prompt = (
    "Summarize its content clearly and objectively in 3–4 concise sentences. "
    "Only include factual information explicitly stated in the transcript, such as financial results, "
    "strategic initiatives, operational updates, or management guidance. "
    "Avoid any interpretation, sentiment, or forward-looking assumptions beyond the given text."
)

In [114]:
df = df.sort_values(by = ['transcriptid', 'componentorder'])

transcripts = df.groupby('transcriptid')['transcript_text'].agg(lambda segs: ' '.join(segs)).reset_index(name='full_transcript')

In [115]:
test = transcripts.iloc[:5]

In [118]:
summaries = []
for _, row in test.iterrows():
    transcript_id = row['transcriptid']
    text = row['full_transcript']
    
    messages = [
        {"role": "system",  "content": Prompt},
        {"role": "user",    "content": text}
    ]
    
    resp = client.chat.completions.create(
        model="gpt-4o",     
        messages=messages,
        max_tokens=500,     
        temperature=0.0
    )
    
    summary = resp.choices[0].message.content.strip()
    summaries.append({
        'transcriptid': transcript_id,
        'summary': summary
    })

In [119]:
summaries

[{'transcriptid': 743348.0,
  'summary': 'Micron Technology reported a record quarterly revenue of $4.6 billion for the first quarter of 2015, with a GAAP net income of $1 billion and free cash flow of $923 million. The company anticipates continued favorable market conditions for 2015, driven by constrained DRAM supply and solid demand for both DRAM and NAND. Micron is focusing on deploying advanced process technology and expects its DRAM production to grow below the market rate due to technology upgrades and product mix optimization. The company is also progressing with its 3D NAND technology, expecting volume production in the second half of 2015, and plans to expand its clean room space in Singapore to support this and other emerging memory technologies.'},
 {'transcriptid': 743536.0,
  'summary': 'During the first quarter of fiscal year 2015, Monsanto reported an ongoing EPS of $0.47, surpassing initial expectations, and a free cash flow of $969 million. The company highlighted si